In [1]:
import numpy as np
from matplotlib import pyplot as plt 
import time 
from tqdm import tqdm

import jax
import jax.numpy as jnp

In [2]:

import jax
from dataclasses import dataclass


## Classes and jitting

In [ ]:

class MyParams:
    def __init__(self, w: jnp.ndarray, b: jnp.ndarray):
        self.w = w
        self.b = b

    def _add(self, array):
        return self.w @ array 

def flatten(params):
    children = (params.w, params.b)
    aux_data = None
    return children, aux_data

def unflatten(aux_data, children):
    w, b = children
    return MyParams(w, b)
 
jax.tree_util.register_pytree_node(MyParams, flatten, unflatten)

MyParams.__tree_flatten__ = flatten
MyParams.__tree_unflatten__ = staticmethod(unflatten)


In [ ]:
# Now you can use it with jit
def update(state: MyParams, a):
    for i in range(300):
        val = state._add(a)
    return val

size = 1000
s = MyParams(jnp.ones((size,size)), jnp.ones((size,size)))
a = jnp.ones((size, size))


start = time.time()
print(update(s, jnp.ones((size,size))))
print(time.time() - start)



In [ ]:
jitted_update = jax.jit(update)
jitted_update(s, jnp.ones((size,size)))

In [ ]:
# Non-JIT
start = time.perf_counter()
out1 = update(s, a).block_until_ready()
end = time.perf_counter()
print(f"Non-JIT time: {end - start:.6f} s")

# JIT (after compilation)
start = time.perf_counter()
out2 = jitted_update(s, a).block_until_ready()
end = time.perf_counter()
print(f"JIT time: {end - start:.6f} s")

# Sanity check: outputs match
print("Outputs close:", jnp.allclose(out1, out2))


In [ ]:
a = jnp.ones((2,2,2))
a_exp = jnp.expand_dims(a, axis = (2,3))
print(a.shape)
print('----')
print(jnp.shape(a_exp))
print((a_exp * jnp.ones(10)).shape)

In [ ]:
print(jnp.arange(0,6))

## Nambu class and jitting

In [3]:
import nambu_class as nmb


In [4]:
s = jnp.ones((300,300))
s_small = jnp.ones(300)
my_nambu_object = nmb.NambuTensor(s)
my_other_object = nmb.NambuTensor(s)
smaller_object = nmb.NambuTensor(s_small)

print(jnp.shape(my_nambu_object.data))
print(jnp.shape(my_other_object.data))




(300, 300)
(300, 300)


In [5]:
my_nambu_object @ my_other_object
my_nambu_object + my_other_object
my_nambu_object - my_other_object
my_nambu_object / my_other_object
my_nambu_object.__matmul__(smaller_object, expansion_indicies = [1])

In [6]:
def run_function(obj1, obj2):
    for i in range(10000):
        result = obj1 @ obj2
        print(jnp.shape(obj1.data), end = ",")
    return 0

In [7]:
print(jnp.shape(my_nambu_object.data))
run_function(my_nambu_object, my_other_object)
print()

(300, 300)
(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300)

In [8]:
print(jnp.shape(my_nambu_object.data))
print(jnp.shape(my_other_object.data))
jitted_run_function = jax.jit(run_function)
jitted_run_function(my_nambu_object, my_other_object)
print()

(300, 300)
(300, 300)
(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300),(300, 300)

In [9]:
jitted_run_function(my_nambu_object,my_other_object)

Array(0, dtype=int32, weak_type=True)

In [10]:
my_new_object = my_nambu_object @ my_other_object

In [11]:
jitted_run_function(my_new_object,my_new_object)

Array(0, dtype=int32, weak_type=True)

In [ ]:
#TODO: Test with different shape tensors this actually compiles using jit 

In [10]:
my_nambu_object / 10

In [ ]:
#TODO: 
# - test all the functions of nambu matrices work properly
# - test if jnp.cos() and similar works on nambu objects
# - check that w-grid only varies w and not theta

In [1]:
import jax 
import jax.numpy as jnp

In [12]:
a = jnp.array([1,2,3])
b = jnp.array([4,5,6])

print(jnp.append(a,b, axis = -1))
print(jnp.stack([a,b], axis = 1).flatten())

[1 2 3 4 5 6]
[1 4 2 5 3 6]
